In [ ]:
import os
import json
import torch
import torch.nn as nn
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from pathlib import Path
from torchvision import transforms
from torchvision.models import efficientnet_b3
from ultralytics import YOLO
from ensemble_boxes import weighted_boxes_fusion

# ==================== НАСТРОЙКИ ====================
DATA_ROOT = "D:/DL_2"
TEST_IMAGES_DIR = "D:/DL_2/test_images/test_images"
SAMPLE_SUB = "D:/DL_2/sample_sub.csv" # Путь к вашему образцу
PREDS_DIR = "D:/DL_2/temp_preds_txt"    # Временная папка для .txt
OUTPUT_CSV = "final_submission.csv"

YOLO_WEIGHTS = [
    "D:/DL_2/yolo_models/yolov8m/weights/best.pt",
    "D:/DL_2/yolo_models/yolov9m/weights/best.pt",
    "D:/DL_2/yolo_models/yolov10m/weights/best.pt"
]
CLASSIFIER_WEIGHTS_PATH = "best_classifier.pth"

# Параметры инференса
CONF_THRESHOLD = 0.15
FINAL_CONF_THRESHOLD = 0.35
WBF_IOU_THR = 0.5
BATCH_SIZE = 16  # Оптимально для RTX 3080 (10-12GB VRAM)
# ==================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(PREDS_DIR, exist_ok=True)

class EfficientNetClassifier(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.backbone = efficientnet_b3(weights=None)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_features, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.35),
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.25),
            nn.Linear(256, num_classes)
        )
    def forward(self, x): return self.backbone(x)

def load_classifier(path):
    model = EfficientNetClassifier(num_classes=2).to(device)
    if os.path.exists(path):
        ckpt = torch.load(path, map_location=device)
        state = ckpt['model_state_dict'] if isinstance(ckpt, dict) and 'model_state_dict' in ckpt else ckpt
        model.load_state_dict(state)
        model.eval()
        print(f"Classifier loaded: {path}")
        return model
    return None

class InferenceEngine:
    def __init__(self, yolo_paths, classifier):
        self.yolo_models = [YOLO(p) for p in yolo_paths]
        self.classifier = classifier
        self.transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

    @torch.no_grad()
    def run(self, image_paths):
        for i in tqdm(range(0, len(image_paths), BATCH_SIZE), desc="Inference"):
            batch_paths = image_paths[i:i+BATCH_SIZE]
            
            # 1. Batch YOLO (Детекция)
            all_yolo_preds = [m(batch_paths, verbose=False, conf=CONF_THRESHOLD) for m in self.yolo_models]

            for b_idx, img_path in enumerate(batch_paths):
                boxes_list, scores_list, labels_list = [], [], []
                
                for m_idx in range(len(self.yolo_models)):
                    res = all_yolo_preds[m_idx][b_idx]
                    if len(res.boxes) > 0:
                        boxes_list.append(res.boxes.xyxyn.cpu().numpy().tolist())
                        scores_list.append(res.boxes.conf.cpu().numpy().tolist())
                        labels_list.append([0] * len(res.boxes))
                    else:
                        boxes_list.append([]); scores_list.append([]); labels_list.append([])

                txt_name = Path(PREDS_DIR) / f"{Path(img_path).stem}.txt"
                
                # 2. WBF (Ансамбль)
                if any(len(b) > 0 for b in boxes_list):
                    f_boxes, f_scores, _ = weighted_boxes_fusion(boxes_list, scores_list, labels_list, iou_thr=WBF_IOU_THR)
                    keep = f_scores >= FINAL_CONF_THRESHOLD
                    f_boxes, f_scores = f_boxes[keep], f_scores[keep]

                    if len(f_boxes) > 0:
                        # Используем уже загруженное изображение из памяти YOLO
                        img_bgr = all_yolo_preds[0][b_idx].orig_img
                        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
                        h_img, w_img = img_rgb.shape[:2]
                        
                        patches = []
                        for box in f_boxes:
                            x1, y1, x2, y2 = int(box[0]*w_img), int(box[1]*h_img), int(box[2]*w_img), int(box[3]*h_img)
                            patch = img_rgb[max(0,y1):min(h_img,y2), max(0,x1):min(w_img,x2)]
                            patches.append(self.transform(Image.fromarray(patch)) if patch.size > 0 else torch.zeros((3,256,256)))
                        
                        # 3. Batch Classify (Классификация всех объектов на фото разом)
                        patch_batch = torch.stack(patches).to(device)
                        preds = self.classifier(patch_batch).argmax(1).cpu().numpy()
                        
                        with open(txt_name, "w") as f:
                            for box, sc, cls in zip(f_boxes, f_scores, preds):
                                w, h = box[2]-box[0], box[3]-box[1]
                                cx, cy = box[0]+w/2, box[1]+h/2
                                f.write(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f} {sc:.6f}\n")
                        continue
                
                # Если людей нет, создаем пустой файл
                txt_name.touch()

# Функция сборки итогового CSV из вашего примера
def build_submission(solution_csv, preds_dir, output_csv):
    sol = pd.read_csv(solution_csv)
    image_names = sol["image_name"].astype(str).tolist()
    rows = []
    
    for idx, name in enumerate(image_names):
        stem = Path(name).stem
        pred_file = Path(preds_dir) / f"{stem}.txt"
        boxes = []
        if pred_file.exists():
            content = pred_file.read_text().strip()
            if content:
                for ln in content.splitlines():
                    p = ln.split()
                    if len(p) >= 6:
                        boxes.append([float(p[1]), float(p[2]), float(p[3]), float(p[4]), float(p[5])])
        rows.append({"id": idx, "image_name": name, "boxes": json.dumps(boxes, separators=(",", ":"))})
    
    pd.DataFrame(rows).to_csv(output_csv, index=False)
    print(f"Submission saved: {output_csv}")

if __name__ == "__main__":
    clf = load_classifier(CLASSIFIER_WEIGHTS_PATH)
    if clf:
        engine = InferenceEngine(YOLO_WEIGHTS, clf)
        test_images = [str(p) for p in Path(TEST_IMAGES_DIR).glob("*") if p.suffix.lower() in ['.jpg', '.jpeg', '.png']]
        
        print(f"Processing {len(test_images)} images...")
        engine.run(test_images)
        
        print("Generating final CSV...")
        build_submission(SAMPLE_SUB, PREDS_DIR, OUTPUT_CSV)
